<a href="https://colab.research.google.com/github/kvikaee/data-analysis-python/blob/main/6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Кошурина Виктория Евгеньевна

412977

PYАД 1.5

# Задание 6. Анализ формы распределений и нормализация данных
### Цели

Научиться:

*	оценивать форму распределений числовых признаков с помощью асимметрии, эксцесса и коэффициента вариации,
*	принимать обоснованные решения о необходимости нормализации/трансформации,
*	применять подходящие методы нормализации
* визуализировать исходные и преобразованные распределения с помощью Plotly


In [ ]:
import numpy as np
import scipy.stats as sps
import pandas as pd
import matplotlib.pyplot as plt
import datetime
from plotly.subplots import make_subplots

import plotly
import plotly.graph_objects as go
import plotly.figure_factory as ff
import plotly.express as px

from tabulate import tabulate

[Ссылка на датасет](https://www.kaggle.com/datasets/neurocipher/student-performance)

В датасете содержатся подробные записи об успеваемости студентов, отражающие множество академических и личностных факторов, влияющих на результаты обучения. Каждая строка представляет студента, а каждый столбец предоставляет информацию о его учебных привычках, образовании и результатах экзаменов.

In [ ]:
df = pd.read_csv('StudentPerformance.csv')

In [ ]:
print(tabulate(
    df.head(5),
    headers='keys',
    tablefmt='simple_grid',
    showindex=False,
    maxcolwidths=20
))

┌─────────────────┬───────────────────┬──────────────────────────────┬───────────────┬────────────────────────────────────┬─────────────────────┐
│   Hours Studied │   Previous Scores │ Extracurricular Activities   │   Sleep Hours │   Sample Question Papers Practiced │   Performance Index │
├─────────────────┼───────────────────┼──────────────────────────────┼───────────────┼────────────────────────────────────┼─────────────────────┤
│               7 │                99 │ Yes                          │             9 │                                  1 │                  91 │
├─────────────────┼───────────────────┼──────────────────────────────┼───────────────┼────────────────────────────────────┼─────────────────────┤
│               4 │                82 │ No                           │             4 │                                  2 │                  65 │
├─────────────────┼───────────────────┼──────────────────────────────┼───────────────┼────────────────────────────────────┼─

### Общая характеристика данных

*	Количество строк и столбцов.
*	Типы признаков (числовые, категориальные).
*	Описание ключевых признаков, особенно категориального с ≥20 значениями.


In [ ]:
num_rows = len(df)  # количество строк
num_columns = len(df.columns)  # количество признаков

# Описание признаков в датасете
columns_description = {
    "Тип данных": df.dtypes,
    "Шкала": [
        "Категориальная" if df[col].dtype == 'object' else
        "Числовая" if df[col].dtype in ['int64', 'float64'] else "Неопределено"
        for col in df.columns
    ]
}

description_df = pd.DataFrame(columns_description)

# Выводим информацию о количестве строк и признаков
print(f"Количество строк: {num_rows}")
print(f"Количество признаков: {num_columns}")

# Строим таблицу с описанием признаков
description_df

Количество строк: 10000
Количество признаков: 6


,Тип данных,Шкала
Hours Studied,int64,Числовая
Previous Scores,int64,Числовая
Extracurricular Activities,object,Категориальная
Sleep Hours,int64,Числовая
Sample Question Papers Practiced,int64,Числовая
Performance Index,float64,Числовая


### Анализ пропусков

Выявление пропущенных значений (включая скрытые, например: "?", "N/A", пустые строки).

In [ ]:
na_counts = df.isna().sum()

# Скрытые маркеры пропусков
hidden_na_values = ["?", "N/A", "NA", "n/a", "NaN", " ", "", "[]"]
mask_hidden = df.isin(hidden_na_values)

hidden_na_counts = mask_hidden.sum()

print("Явные пропуски:\n", na_counts)
print("\nСкрытые пропуски:\n", hidden_na_counts)

Явные пропуски:
 Hours Studied                       0
Previous Scores                     0
Extracurricular Activities          0
Sleep Hours                         0
Sample Question Papers Practiced    0
Performance Index                   0
dtype: int64

Скрытые пропуски:
 Hours Studied                       0
Previous Scores                     0
Extracurricular Activities          0
Sleep Hours                         0
Sample Question Papers Practiced    0
Performance Index                   0
dtype: int64


## Краткий анализ распределения
(визуализация)

In [ ]:
numeric_cols = df.select_dtypes(include=['number']).columns.tolist()

In [ ]:
numeric_cols_filtered = [col for col in numeric_cols if col]

fig = make_subplots(
    rows=len(numeric_cols_filtered),
    cols=1,
    subplot_titles=[f'Распределение {col}' for col in numeric_cols_filtered],
    vertical_spacing=0.08
)

for idx, col in enumerate(numeric_cols_filtered, 1):
    data = df[col].dropna()

    # Гистограмма с нормализацией
    fig.add_trace(
        go.Histogram(
            x=data,
            nbinsx=30,
            name='распределение',
            marker_color='steelblue',
            opacity=0.6,
            showlegend=(idx == 1),
            histnorm='density'
        ),
        row=idx, col=1
    )

    # Метки осей
    fig.update_yaxes(title_text='Плотность', row=idx, col=1)
    fig.update_xaxes(title_text=col, row=idx, col=1)

fig.update_layout(
    height=300 * len(numeric_cols_filtered),
    title_text='Распределения числовых переменных',
    hovermode='x unified'
)

fig.show()


In [ ]:
# Категориальный признак
activity_counts = df['Extracurricular Activities'].value_counts()

# Создаем pie chart
fig = go.Figure(data=[go.Pie(
    labels=activity_counts.index,
    values=activity_counts.values,
    marker=dict(line=dict(color='white', width=3))
)])

# Форматирование
fig.update_layout(
    title='Распределение внеклассной деятельности студентов  (Yes - есть внеклассная активность, No - отсутствует)',
    font=dict(size=12),
    height=600,
    width=1000,
    hovermode='closest'
)

# Добавляем процент в подсказку
fig.update_traces(
    textposition='inside',
    textinfo='label+percent',
    hovertemplate='<b>%{label}</b><br>Количество: %{value}<br>Процент: %{percent}<extra></extra>'
)

fig.show()



## Часть 1. Анализ исходных распределений
Для каждого числового признака рассчитайте:

*	среднее, медианы, моды (что целесообразно рассматривать?), стандартное отклонение,
*	коэффициент асимметрии (scipy.stats.skew),
*	избыточный эксцесс (scipy.stats.kurtosis(..., fisher=True)),
*	коэффициент вариации.


In [ ]:
def calculate_statistics(df, numeric_cols):
    """Рассчитать полную статистику для числовых признаков"""

    stats_dict = {}

    for col in numeric_cols:
        data = df[col].dropna()

        # Базовая статистика
        mean_val = data.mean()
        median_val = data.median()
        mode_val = data.mode().values[0] if len(data.mode()) > 0 else np.nan
        std_val = data.std()

        # Асимметрия и эксцесс
        skewness = sps.skew(data)
        excess_kurtosis = sps.kurtosis(data, fisher=True)

        # Коэффициент вариации (%)
        cv = (std_val / mean_val * 100) if mean_val != 0 else np.nan

        # Минимум и максимум
        min_val = data.min()
        max_val = data.max()

        stats_dict[col] = {
            'Mean': mean_val,
            'Median': median_val,
            'Mode': mode_val,
            'Std Dev': std_val,
            'Min': min_val,
            'Max': max_val,
            'Skewness': skewness,
            'Excess Kurtosis': excess_kurtosis,
            'Coefficient of Variation (%)': cv
        }

    return stats_dict

# Рассчитываем статистику
stats_dict = calculate_statistics(df, numeric_cols)


In [ ]:
stats_df = pd.DataFrame(stats_dict).T
stats_display = stats_df.round(4)

# Используем tabulate для красивого вывода
table = tabulate(
    stats_display,
    headers=stats_display.columns,
    tablefmt='fancy_grid',
    floatfmt='.4f',
    showindex=True
)

print(table)

╒══════════════════════════════════╤═════════╤══════════╤═════════╤═══════════╤═════════╤══════════╤════════════╤═══════════════════╤════════════════════════════════╕
│                                  │    Mean │   Median │    Mode │   Std Dev │     Min │      Max │   Skewness │   Excess Kurtosis │   Coefficient of Variation (%) │
╞══════════════════════════════════╪═════════╪══════════╪═════════╪═══════════╪═════════╪══════════╪════════════╪═══════════════════╪════════════════════════════════╡
│ Hours Studied                    │  4.9929 │   5.0000 │  1.0000 │    2.5893 │  1.0000 │   9.0000 │    -0.0042 │           -1.2321 │                        51.8598 │
├──────────────────────────────────┼─────────┼──────────┼─────────┼───────────┼─────────┼──────────┼────────────┼───────────────────┼────────────────────────────────┤
│ Previous Scores                  │ 69.4457 │  69.0000 │ 54.0000 │   17.3432 │ 40.0000 │  99.0000 │     0.0045 │           -1.2132 │                        24.9737 

In [ ]:
def interpret_statistics(stats_dict):
    """Интерпретировать статистики для каждого признака"""

    print("\n" + "="*100)
    print("ИНТЕРПРЕТАЦИЯ РАСПРЕДЕЛЕНИЙ")
    print("="*100)

    for col, stats_vals in stats_dict.items():
        print(f"\n {col.upper()}")
        print("-" * 100)

        skew = stats_vals['Skewness']
        kurt = stats_vals['Excess Kurtosis']
        cv = stats_vals['Coefficient of Variation (%)']

        # Интерпретация асимметрии
        print(f"  • Асимметрия (Skewness): {skew:.4f}", end="")
        if abs(skew) < 0.5:
            print(" → Близко к симметричному распределению")
        elif 0.5 <= abs(skew) < 1:
            print(" → Умеренная асимметрия")
        elif abs(skew) >= 1:
            if skew > 0:
                print(" → !!! СИЛЬНАЯ ПРАВОСТОРОННЯЯ асимметрия (хвост вправо)")
            else:
                print(" → !!! СИЛЬНАЯ ЛЕВОСТОРОННЯЯ асимметрия (хвост влево)")

        # Интерпретация эксцесса
        print(f"  • Избыточный эксцесс (Kurtosis): {kurt:.4f}", end="")
        if abs(kurt) < 0.5:
            print(" → Похожее на нормальное распределение")
        elif 0.5 <= abs(kurt) < 1:
            print(" → Умеренный эксцесс")
        elif abs(kurt) >= 1:
            if kurt > 0:
                print(" → !!! ВЫСОКИЙ ПОЛОЖИТЕЛЬНЫЙ эксцесс (тяжёлые хвосты, острый пик)")
            else:
                print(" → !!! ОТРИЦАТЕЛЬНЫЙ эксцесс (плоское распределение)")

        # Интерпретация коэффициента вариации
        print(f"  • Коэффициент вариации: {cv:.2f}%", end="")
        if cv < 15:
            print(" → Низкая вариабельность (стабильные данные)")
        elif 15 <= cv < 30:
            print(" → Средняя вариабельность")
        elif 30 <= cv < 50:
            print(" → Высокая вариабельность")
        else:
            print(" → !!! ОЧЕНЬ ВЫСОКАЯ вариабельность (нестабильные данные)")

        # Отличие от нормального распределения
        print(f"\n  ОТЛИЧИЕ ОТ НОРМАЛЬНОГО:")
        normality_score = abs(skew) + abs(kurt)
        if normality_score < 1:
            print(f"     Распределение близко к нормальному (score: {normality_score:.2f})")
        elif normality_score < 2:
            print(f"     Умеренное отличие от нормального (score: {normality_score:.2f})")
        else:
            print(f"     !!! ЗНАЧИТЕЛЬНОЕ отличие от нормального (score: {normality_score:.2f})")

# Выводим интерпретацию
interpret_statistics(stats_dict)



ИНТЕРПРЕТАЦИЯ РАСПРЕДЕЛЕНИЙ

 HOURS STUDIED
----------------------------------------------------------------------------------------------------
  • Асимметрия (Skewness): -0.0042 → Близко к симметричному распределению
  • Избыточный эксцесс (Kurtosis): -1.2321 → !!! ОТРИЦАТЕЛЬНЫЙ эксцесс (плоское распределение)
  • Коэффициент вариации: 51.86% → !!! ОЧЕНЬ ВЫСОКАЯ вариабельность (нестабильные данные)

  ОТЛИЧИЕ ОТ НОРМАЛЬНОГО:
     Умеренное отличие от нормального (score: 1.24)

 PREVIOUS SCORES
----------------------------------------------------------------------------------------------------
  • Асимметрия (Skewness): 0.0045 → Близко к симметричному распределению
  • Избыточный эксцесс (Kurtosis): -1.2132 → !!! ОТРИЦАТЕЛЬНЫЙ эксцесс (плоское распределение)
  • Коэффициент вариации: 24.97% → Средняя вариабельность

  ОТЛИЧИЕ ОТ НОРМАЛЬНОГО:
     Умеренное отличие от нормального (score: 1.22)

 SLEEP HOURS
------------------------------------------------------------------------------

### Часть 2. Визуализация (Plotly)

*	Создайте интерактивные гистограммы, ящики с усами + кривые плотности для всех числовых признаков.
*	Используйте plotly.subplots для компактного отображения.
*	На каждом графике добавьте аннотацию с численными значениями:
Skew: ..., Kurtosis: ..., CV: ...%


In [ ]:
from scipy.stats import gaussian_kde

In [ ]:
fig = make_subplots(
    rows=len(numeric_cols_filtered),
    cols=3,
    subplot_titles=[
        item for col in numeric_cols_filtered
        for item in [f'Гистограмма {col}', f'Ящик {col}', f'KDE {col}']
    ],
    vertical_spacing=0.1,
    horizontal_spacing=0.12
)

for idx, col in enumerate(numeric_cols_filtered, 1):
    data = df[col].dropna()

    # Получаем статистику
    skew = stats_dict[col]['Skewness']
    kurt = stats_dict[col]['Excess Kurtosis']
    cv = stats_dict[col]['Coefficient of Variation (%)']
    annotation_text = f"Skew: {skew:.3f}<br>Kurtosis: {kurt:.3f}<br>CV: {cv:.1f}%"

    # 1. Гистограмма
    fig.add_trace(
        go.Histogram(
            x=data,
            nbinsx=30,
            marker_color='steelblue',
            opacity=0.7,
            showlegend=False
        ),
        row=idx, col=1
    )

    # 2. Ящик с усами
    fig.add_trace(
        go.Box(
            y=data,
            marker_color='lightgreen',
            showlegend=False
        ),
        row=idx, col=2
    )

    # 3. KDE кривая плотности
    kde = gaussian_kde(data)
    x_range = np.linspace(data.min(), data.max(), 200)
    kde_values = kde(x_range)

    fig.add_trace(
        go.Scatter(
            x=x_range,
            y=kde_values,
            mode='lines',
            line=dict(color='red', width=2),
            fill='tozeroy',
            showlegend=False
        ),
        row=idx, col=3
    )

    # Аннотация со статистикой
    fig.add_annotation(
        text=annotation_text,
        xref="paper", yref="paper",
        x=0.01, y=1 - (idx - 0.5) / len(numeric_cols_filtered),
        showarrow=False,
        bgcolor="rgba(255, 255, 200, 0.95)",
        bordercolor="gray",
        borderwidth=1,
        borderpad=8,
        font=dict(size=9),
        xanchor="left",
        yanchor="middle"
    )

    # Метки осей
    fig.update_xaxes(title_text=col, row=idx, col=1)
    fig.update_xaxes(title_text=col, row=idx, col=3)
    fig.update_yaxes(title_text='Частота', row=idx, col=1)
    fig.update_yaxes(title_text='Значение', row=idx, col=2)
    fig.update_yaxes(title_text='Плотность', row=idx, col=3)

fig.update_layout(
    height=300 * len(numeric_cols_filtered),
    width=1600,
    title_text='Анализ распределений всех числовых признаков',
    hovermode='closest'
)

fig.show()

### Часть 3. Нормализация / трансформация

*	Выберите 2–3 признака с сильной асимметрией или высоким CV.
*	Примените не менее двух методов трансформации, например:
*	Логарифмическая (np.log1p),
*	Квадратный корень,
*	Box-Cox (только для строго положительных данных),
*	Или масштабирование через RobustScaler/PowerTransformer.
*	Для каждого преобразования:
*	Пересчитайте skew, kurtosis, CV.
*	Постройте сравнительные графики «до и после» в Plotly (например, с помощью make_subplots).


In [ ]:
selected_cols = [
    'Hours Studied',
    'Sample Question Papers Practiced',
    'Performance Index'
]

In [ ]:
def compute_stats(values):
    # приводим к numpy-массиву и убираем NaN
    arr = np.asarray(values)
    arr = arr[~np.isnan(arr)]

    mean_val = arr.mean()
    std_val = arr.std(ddof=1)  # как в pandas
    skewness = sps.skew(arr)
    excess_kurtosis = sps.kurtosis(arr, fisher=True)
    cv = (std_val / mean_val * 100) if mean_val != 0 else np.nan

    return {
        'Mean': mean_val,
        'Std Dev': std_val,
        'Skewness': skewness,
        'Excess Kurtosis': excess_kurtosis,
        'Coefficient of Variation (%)': cv
    }



In [ ]:
from scipy.stats import boxcox

In [ ]:
all_stats = {}  # сюда сложим таблицы со статистикой для отчёта

for col in selected_cols:
    data = df[col].dropna().astype(float)

    # трансформации
    transformed = {}
    transformed['Original'] = data

    # 1) log1p
    transformed['Log1p'] = np.log1p(data)

    # 2) sqrt
    transformed['Sqrt'] = np.sqrt(data)

    # 3) Box-Cox (обеспечиваем строго > 0)
    shift = 0.0
    if (data <= 0).any():
        shift = 1.0 - data.min()
    bc_data, bc_lambda = boxcox(data + shift)
    transformed['Box-Cox'] = bc_data

    # статистики для всех вариантов
    stats_for_col = {name: compute_stats(ser) for name, ser in transformed.items()}
    all_stats[col] = pd.DataFrame(stats_for_col).T

    #  2×2 subplot: Original, Log1p, Sqrt, Box-Cox
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=[
            f'{col} — Original',
            f'{col} — Log1p',
            f'{col} — Sqrt',
            f'{col} — Box-Cox (λ={bc_lambda:.2f})'
        ],
        vertical_spacing=0.18,
        horizontal_spacing=0.12
    )

    # порядок следования в subplot
    names_order = ['Original', 'Log1p', 'Sqrt', 'Box-Cox']
    positions = {
        1: (1, 1),
        2: (1, 2),
        3: (2, 1),
        4: (2, 2)
    }
    # координаты аннотаций в единицах "paper", чтобы ничего не съезжало
    ann_positions = {
        1: (0.22, 0.93),
        2: (0.78, 0.93),
        3: (0.22, 0.38),
        4: (0.78, 0.38)
    }

    for i, name in enumerate(names_order, start=1):
        row, col_pos = positions[i]
        ser = transformed[name]

        fig.add_trace(
            go.Histogram(
                x=ser,
                nbinsx=30,
                marker_color='steelblue',
                opacity=0.75,
                showlegend=False
            ),
            row=row, col=col_pos
        )

        # аннотация Skew, Kurtosis, CV для каждого окна
        st = stats_for_col[name]
        ann_text = (
            f"Skew: {st['Skewness']:.3f}<br>"
            f"Kurtosis: {st['Excess Kurtosis']:.3f}<br>"
            f"CV: {st['Coefficient of Variation (%)']:.1f}%"
        )
        ax, ay = ann_positions[i]
        fig.add_annotation(
            text=ann_text,
            xref="paper", yref="paper",
            x=ax, y=ay,
            showarrow=False,
            bgcolor="rgba(255,255,200,0.95)",
            bordercolor="gray",
            borderwidth=1,
            borderpad=6,
            font=dict(size=10),
            xanchor="center",
            yanchor="top"
        )

    fig.update_xaxes(title_text='Значение', row=1, col=1)
    fig.update_xaxes(title_text='Значение', row=1, col=2)
    fig.update_xaxes(title_text='Значение', row=2, col=1)
    fig.update_xaxes(title_text='Значение', row=2, col=2)

    fig.update_yaxes(title_text='Частота')

    fig.update_layout(
        height=700,
        width=1100,
        title_text=f'Нормализация и сравнение распределений: {col}',
        bargap=0.05
    )

    fig.show()

    # при необходимости можно ещё вывести таблицу со статистикой
    display(all_stats[col].round(4))

,Mean,Std Dev,Skewness,Excess Kurtosis,Coefficient of Variation (%)
Original,4.9929,2.5893,-0.0042,-1.2321,51.8598
Log1p,1.6759,0.5087,-0.6097,-0.7727,30.3551
Sqrt,2.1425,0.6344,-0.3865,-1.0071,29.6080
Box-Cox,2.8954,1.7209,-0.2081,-1.1499,59.4372


,Mean,Std Dev,Skewness,Excess Kurtosis,Coefficient of Variation (%)
Original,4.5833,2.8673,-0.0356,-1.2145,62.5608
Log1p,1.5301,0.6890,-0.9202,-0.1473,45.0326
Sqrt,1.9555,0.8715,-0.8717,-0.0615,44.5645
Box-Cox,3.3971,1.9570,-0.2180,-1.1261,57.6077


,Mean,Std Dev,Skewness,Excess Kurtosis,Coefficient of Variation (%)
Original,55.2248,19.2126,-0.0017,-0.8607,34.7897
Log1p,3.9613,0.3880,-0.7639,0.1863,9.7947
Sqrt,7.3075,1.3513,-0.3550,-0.5861,18.4916
Box-Cox,34.7912,10.8923,-0.0969,-0.8228,31.3077


**Hours Studied.** Исходное распределение почти симметрично, но с высоким коэффициентом вариации и заметным положительным эксцессом, что говорит о большом разбросе и «остром» пике. Логарифмическая и особенно корневая трансформации существенно снижают CV (почти вдвое) и сглаживают хвосты, приближая распределение к нормальному при умеренной отрицательной асимметрии.

**Performance Index.** В исходном виде признак уже близок к нормальному распределению: асимметрия практически нулевая, а эксцесс лишь немного выше нуля. Корневая и Box‑Cox‑трансформации лишь слегка уменьшают вариабельность и делают распределение чуть более «плоским» и симметричным, поэтому их использование носит скорее косметический характер.

**Sample Question Papers Practiced.** Для целочисленного признака исходное распределение отличается очень высокой вариабельностью и острым пиком при почти нулевой асимметрии. Логарифмическая и корневая трансформации заметно снижают CV и эксцесс, делая хвосты легче, но при этом вводят выраженную левостороннюю асимметрию; Box‑Cox даёт более симметричную форму, однако хуже справляется с уменьшением разброса.

### Часть 4. Вывод

*	Сформулируйте, какие трансформации оказались наиболее эффективными и почему.
*	Обоснуйте, стоит ли применять эти преобразования перед обучением ML-моделей на этом датасете.


**Какие трансформации оказались наиболее эффективными**

Можно выделить следующие закономерности:

- Для признаков **Hours Studied** и **Sample Question Papers Practiced** наиболее эффективными оказались **логарифмическая и корневая трансформации**. Они заметно снижают коэффициент вариации и избыточный эксцесс (то есть уменьшают разброс и «остроту» пика), делая распределения более компактными и с менее тяжёлыми хвостами. Box-Cox в этих случаях даёт более симметричную форму, но хуже справляется с уменьшением вариабельности, поэтому его эффект менее выгоден с практической точки зрения.

- Для **Performance Index** исходное распределение уже близко к нормальному, с малой асимметрией и умеренным эксцессом. Корневая и Box-Cox-трансформации лишь немного сглаживают хвосты и снижают CV, то есть дают скорее косметическое улучшение. Логарифмическое преобразование здесь излишне агрессивно: оно заметно искажает форму распределения (сильная левая асимметрия) при отсутствии серьёзной проблемы в исходных данных.

В целом, можно считать, что для «шумных» дискретных признаков (часы и количество решённых бумаг) полезнее всего мягкие монотонные трансформации типа *log1p* / *sqrt*, а для уже почти нормального показателя успеваемости преобразования необязательны.



**Нужно ли применять трансформации перед обучением ML-моделей**

Здесь надо ориентироваться на тип используемых моделей:

- Для моделей, чувствительных к масштабу и форме распределений, применение трансформаций выглядит обоснованным. В таком случае:  
  1) применить *log1p* или *sqrt* к **Hours Studied** и **Sample Question Papers Practiced** (где трансформации заметно улучшают распределения и снижают влияние выбросов);  
  2) оставить **Performance Index** в исходном виде или использовать для него мягкую *sqrt*‑трансформацию;  

Итого: трансформации *log1p* и *sqrt* для признаков Hours Studied и Sample Question Papers Practiced заметно уменьшают вариабельность и делают распределения ближе к нормальному, поэтому их целесообразно применять перед обучением моделей, чувствительных к масштабу и форме распределений признаков. Для *Performance Index* эффект трансформаций минимален, поэтому этот признак можно оставлять в исходном виде.